# Section 4 — EDA Figures
**Purpose**: Generate the 3 required EDA figures for the CISC 886 report.

> CPU runtime is fine. No GPU needed.

In [ ]:
!pip install -q boto3 matplotlib seaborn
import boto3, json, os, matplotlib.pyplot as plt, seaborn as sns
from collections import Counter
from getpass import getpass

aws_key    = getpass('AWS Access Key ID: ')
aws_secret = getpass('AWS Secret Access Key: ')
s3 = boto3.client('s3', aws_access_key_id=aws_key, aws_secret_access_key=aws_secret, region_name='us-east-1')
BUCKET = '25fltp-ecom-chatbot'
os.makedirs('./data', exist_ok=True)

# Download all 3 splits
splits = {'train': 0, 'val': 0, 'test': 0}
for split in splits:
    path = f'./data/{split}.jsonl'
    s3.download_file(BUCKET, f'processed/{split}.jsonl/part-00000', path)
    splits[split] = sum(1 for _ in open(path))
    print(f'✅ {split}: {splits[split]:,} samples')

# Load a sample for detailed analysis
records = []
with open('./data/train.jsonl') as f:
    for i, line in enumerate(f):
        if i >= 5000: break
        try: records.append(json.loads(line))
        except: pass
print(f'\nLoaded {len(records)} records for EDA')

## Figure 1: Token Length Distribution

In [ ]:
lengths = [len(str(r.get('text', r)).split()) for r in records]
mean_len = sum(lengths) / len(lengths)

plt.figure(figsize=(9, 4))
plt.hist(lengths, bins=40, color='#3A6EA5', edgecolor='white', alpha=0.85)
plt.axvline(mean_len, color='#E74C3C', linestyle='--', linewidth=2, label=f'Mean: {mean_len:.0f} tokens')
plt.title('Figure 1: Token Length Distribution\n(Training Split — 5,000 samples)', fontweight='bold')
plt.xlabel('Tokens per Sample')
plt.ylabel('Frequency')
plt.legend()
plt.tight_layout()
plt.savefig('fig1_token_length.png', dpi=150, bbox_inches='tight')
plt.show()
print('Caption: Token lengths are right-skewed with most samples under 300 tokens, well within the 2048 max context window used during fine-tuning.')

## Figure 2: Sentiment Class Balance

In [ ]:
sentiments = [r.get('sentiment_spark', 'unknown') for r in records if 'sentiment_spark' in r]
counts = Counter(sentiments)

color_map = {'positive': '#2ECC71', 'neutral': '#F39C12', 'negative': '#E74C3C', 'unknown': '#95A5A6'}
labels = list(counts.keys())
values = list(counts.values())
colors = [color_map.get(l, '#95A5A6') for l in labels]

plt.figure(figsize=(7, 4))
bars = plt.bar(labels, values, color=colors, edgecolor='white', width=0.5)
for b in bars:
    plt.text(b.get_x() + b.get_width()/2, b.get_height() + 30,
             f'{int(b.get_height()):,}', ha='center', fontsize=11, fontweight='bold')
plt.title('Figure 2: Sentiment Class Balance\n(Training Split — 5,000 samples)', fontweight='bold')
plt.xlabel('Sentiment Class')
plt.ylabel('Sample Count')
plt.tight_layout()
plt.savefig('fig2_sentiment_balance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Caption: Positive reviews dominate (rating >= 4), reflecting the general customer satisfaction in the Amazon dataset. Class imbalance is noted for future mitigation.')

## Figure 3: Sample Count per Split

In [ ]:
split_labels = list(splits.keys())
split_counts = list(splits.values())
total = sum(split_counts)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

# Pie chart
ax1.pie(split_counts, labels=[f'{l.capitalize()}\n{c:,}' for l,c in zip(split_labels, split_counts)],
        autopct='%1.1f%%', colors=['#3A6EA5','#E67E22','#27AE60'],
        wedgeprops={'edgecolor':'white','linewidth':2})
ax1.set_title('Split Distribution (Pie)', fontweight='bold')

# Bar chart
bars = ax2.bar([l.capitalize() for l in split_labels], split_counts,
               color=['#3A6EA5','#E67E22','#27AE60'], edgecolor='white')
for b in bars:
    ax2.text(b.get_x()+b.get_width()/2, b.get_height()+1000,
             f'{int(b.get_height()):,}', ha='center', fontweight='bold')
ax2.set_title('Split Distribution (Count)', fontweight='bold')
ax2.set_ylabel('Number of Samples')

plt.suptitle(f'Figure 3: Sample Count per Dataset Split  (Total: {total:,})', fontweight='bold', fontsize=12)
plt.tight_layout()
plt.savefig('fig3_split_counts.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Caption: Dataset uses an 80/10/10 train/validation/test split across {total:,} total samples.')
print('\n✅ All 3 EDA figures saved! Download them for your report.')